In [ ]:
import os
import re
import pandas as pd

def count_line_types(file_path):
    code_lines = 0
    comment_lines = 0
    empty_lines = 0
    in_block_comment = False

    with open(file_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                empty_lines += 1
                continue

            if in_block_comment:
                comment_lines += 1
                if '*/' in stripped:
                    in_block_comment = False
                continue

            if stripped.startswith('/*'):
                comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if stripped.startswith('//'):
                comment_lines += 1
                continue

            if '/*' in stripped:
                before_comment = stripped.split('/*', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                if not '*/' in stripped:
                    in_block_comment = True
                continue

            if '//' in stripped:
                before_comment = stripped.split('//', 1)[0].strip()
                if before_comment:
                    code_lines += 1
                else:
                    comment_lines += 1
                continue

            code_lines += 1

    return code_lines, comment_lines, empty_lines

def count_test_methods(content):
    test_annotation_pattern = re.compile(
        r'@(?:org\.junit(?:\.jupiter\.api)?\.)?(Test|RepeatedTest|ParameterizedTest)\b'
    )
    method_pattern = re.compile(
        r'\b\w[\w<>\[\],\s]*\s+\w+\s*\('
    )

    lines = content.splitlines()
    num_test_methods = 0
    pending_test_method = False
    method_decl = ''
    for line in lines:
        stripped = line.strip()
        # Ignore single-line comments and JavaDoc lines
        if stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
            continue

        # If annotation and method on the same line
        if test_annotation_pattern.search(stripped) and method_pattern.search(stripped):
            num_test_methods += 1
            pending_test_method = False
            method_decl = ''
            continue

        # If we see a test annotation, set the flag
        if test_annotation_pattern.search(stripped):
            pending_test_method = True
            method_decl = ''
            continue

        # If we're waiting for a method after a test annotation
        if pending_test_method:
            # Skip blank lines and annotation lines
            if not stripped or stripped.startswith('@') or stripped.startswith('//') or stripped.startswith('*') or stripped.startswith('/*') or stripped.startswith('*/'):
                continue
            # Accumulate lines for multi-line method declarations
            method_decl += ' ' + stripped
            if '(' in method_decl:
                if method_pattern.search(method_decl):
                    num_test_methods += 1
                    pending_test_method = False
                    method_decl = ''
            continue

        # Reset method_decl if not in pending state
        method_decl = ''
    return num_test_methods

def collect_stats(directory):
    num_files = 0
    num_classes = 0
    num_code_lines = 0
    num_comment_lines = 0
    num_empty_lines = 0
    num_test_methods = 0

    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".java"):
                num_files += 1
                file_path = os.path.join(root, file)
                code, comment, empty = count_line_types(file_path)
                num_code_lines += code
                num_comment_lines += comment
                num_empty_lines += empty

                with open(file_path, encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                    num_classes += len(re.findall(r'\b(class|interface|enum)\s+\w+', content))
                    num_test_methods += count_test_methods(content)

    return {
        "directory": directory,
        "num_files": num_files,
        "num_classes": num_classes,
        "num_code_lines": num_code_lines,
        "num_comment_lines": num_comment_lines,
        "num_empty_lines": num_empty_lines,
        "num_test_methods": num_test_methods,
    }

# List of directories to analyze
directories = [
    "../projects/commons-utils/src/main",
    "../projects/commons-utils/src/test",
    "../projects/commons-utils-es-default-1s/src/main",
    "../projects/commons-utils-es-default-1s/src/test",
    "../projects/commons-utils-es-default-10s/src/main",
    "../projects/commons-utils-es-default-10s/src/test",
    "../projects/commons-utils-es-default-60s/src/main",
    "../projects/commons-utils-es-default-60s/src/test",
    "../projects/eqbench-es-default-1s/src/main",
    "../projects/eqbench-es-default-1s/src/test",
    "../projects/eqbench-es-default-10s/src/main",
    "../projects/eqbench-es-default-10s/src/test",
    "../projects/eqbench-es-default-60s/src/main",
    "../projects/eqbench-es-default-60s/src/test",
]

# Helper to extract project name from directory path
def get_project_name(directory):
    # Assumes project is the last segment before 'src'
    parts = directory.split(os.sep)
    for i, part in enumerate(parts):
        if part == "src" and i > 0:
            return parts[i-1]
    return directory

# Collect statistics for all directories
stats = []
for d in directories:
    stat = collect_stats(d)
    stat["project"] = get_project_name(d)
    stat["type"] = "main" if d.endswith("/main") else "test"
    stats.append(stat)

df = pd.DataFrame(stats)
display(df)

# Aggregate by project
summary = []
for project in df["project"].unique():
    main = df[(df["project"] == project) & (df["type"] == "main")]
    test = df[(df["project"] == project) & (df["type"] == "test")]
    summary.append({
        "project": project,
        "main_files": int(main["num_files"].sum()),
        "main_classes": int(main["num_classes"].sum()),
        "main_sloc": int(main["num_code_lines"].sum()),
        "test_files": int(test["num_files"].sum()),
        "test_classes": int(test["num_classes"].sum()),
        "test_sloc": int(test["num_code_lines"].sum()),
        "test_methods": int(test["num_test_methods"].sum()),
    })

df_summary = pd.DataFrame(summary)
display(df_summary)

# Test method counts should match:
# """
# SELECT project_name(project_id), count(*)
# FROM test t JOIN project p ON t.project_id = p.id
# WHERE p.use_test_generalization
# GROUP BY project_id
# """

def get_base_project(name):
    return name.split('-es-')[0]

# Build LaTeX table
latex_table = r"""\begin{table}[H]
  \caption{Number of files, classes, lines, and test methods per project.}
  \label{tab:dataset-statistics}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & \multicolumn{3}{c}{Implementation} & \multicolumn{4}{c}{Test} \\
    \cmidrule(lr){2-4} \cmidrule(lr){5-8}
    Project & Files & Classes & Lines & Files & Classes & Lines & Methods \\
    \midrule
"""

prev_base = None
for _, row in df_summary.iterrows():
    base = get_base_project(row["project"])
    if prev_base is not None and base != prev_base:
        latex_table += "    \\midrule\n"
    prev_base = base
    latex_table += (
        f"    {row['project']} & "
        f"{row['main_files']} & {row['main_classes']} & {row['main_sloc']} & "
        f"{row['test_files']} & {row['test_classes']} & {row['test_sloc']} & {row['test_methods']} \\\\\n"
    )

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)